In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# 1) Load and balance the full dataset to exactly 339 of each class
df = pd.read_csv('ai4i2020.csv')
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# down-sample via pandas
df_min = df[df['Machine failure'] == 1].sample(n=339, random_state=42)
df_maj = df[df['Machine failure'] == 0].sample(n=339, random_state=42)
df_bal = pd.concat([df_min, df_maj]).sample(frac=1, random_state=42)

X_bal = df_bal[feature_cols]
y_bal = df_bal['Machine failure']

# 2) Split into TRAIN (80%) and TEST (20%)—test remains untouched
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)  # https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

# 3) Build preprocessor for numeric scaling
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), feature_cols)
])

# 4) Define Matthews Correlation Coefficient as scoring metric
mcc_scorer = make_scorer(matthews_corrcoef)

# 5) Define pipelines and hyperparameter grids for each model
models = {
    "MLP": (
        Pipeline([("scale", preprocessor), ("clf", MLPClassifier(max_iter=500, random_state=42))]),
        {
            "clf__hidden_layer_sizes": [(50,), (100,), (50, 50)],
            "clf__activation": ["relu", "tanh"],
            "clf__learning_rate": ["constant", "adaptive"]
        }
    ),
    "SVM": (
        Pipeline([("scale", preprocessor), ("clf", SVC(random_state=42))]),
        {
            "clf__C": [0.1, 1, 10],
            "clf__kernel": ["linear", "rbf"],
            "clf__gamma": ["scale", "auto"]
        }
    ),
    "KNN": (
        Pipeline([("scale", preprocessor), ("clf", KNeighborsClassifier())]),
        {
            "clf__n_neighbors": [3, 5, 7],
            "clf__p": [1, 2],
            "clf__algorithm": ["auto", "ball_tree"]
        }
    ),
    "DecisionTree": (
        Pipeline([("scale", preprocessor), ("clf", DecisionTreeClassifier(random_state=42))]),
        {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_depth": [None, 5, 10],
            "clf__ccp_alpha": [0.0, 0.01, 0.1]
        }
    ),
    "LogisticRegression": (
        Pipeline([("scale", preprocessor), ("clf", LogisticRegression(random_state=42, solver="liblinear"))]),
        {
            "clf__penalty": ["l2", "l1"],
            "clf__C": [0.1, 1, 10],
            "clf__solver": ["liblinear"]
        }
    )
}

# 6) Perform 5‑fold GridSearchCV on the TRAINING set for each model
best_estimators = {}
for name, (pipeline, params) in models.items():
    print(f"Tuning {name}...")
    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=mcc_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    gs.fit(X_train, y_train)
    print(f"Best {name} CV MCC: {gs.best_score_:.4f}")
    print(f"Best {name} params: {gs.best_params_}\n")
    best_estimators[name] = gs.best_estimator_

# best_estimators now contains the tuned pipeline for each model.



Tuning MLP...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't conver

Best MLP CV MCC: 0.8068
Best MLP params: {'clf__activation': 'relu', 'clf__hidden_layer_sizes': (50, 50), 'clf__learning_rate': 'constant'}

Tuning SVM...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best SVM CV MCC: 0.8204
Best SVM params: {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}

Tuning KNN...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best KNN CV MCC: 0.7518
Best KNN params: {'clf__algorithm': 'auto', 'clf__n_neighbors': 5, 'clf__p': 1}

Tuning DecisionTree...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best DecisionTree CV MCC: 0.8310
Best DecisionTree params: {'clf__ccp_alpha': 0.01, 'clf__criterion': 'gini', 'clf__max_depth': 5}

Tuning LogisticRegression...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best LogisticRegression CV MCC: 0.6514
Best LogisticRegression params: {'clf__C': 1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

